<a href="https://colab.research.google.com/github/owcaelfis6626/test/blob/main/scratchpad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
cov = (h_interp * needed_correction).mean().item()
corr = cov / (h_interp.std().item() * needed_correction.std().item())
print("Correlation between h_interp and needed_correction:", corr)

# What if h had opposite sign?
diff_neg = (-h_interp - needed_correction)
print("If h had opposite sign:")
print("  rms:", diff_neg.square().mean().sqrt().item())
print("  rel error:", diff_neg.square().mean().sqrt().item() / ref_sel.square().mean().sqrt().item())


Correlation between h_interp and needed_correction: 0.042717435552468395
If h had opposite sign:
  rms: 0.08729117778921866
  rel error: 0.6193413293985078


In [23]:
!mkdir -p /content/benchmark
!curl -s -o /content/benchmark/test.pt https://litter.catbox.moe/lxq3vs.pt
!curl -s -o /content/benchmark/n2049_f4_values.pt https://litter.catbox.moe/32sy5y.pt
!curl -s -o /content/benchmark/baseline_model.pt https://litter.catbox.moe/i6pmzw.pt

!sha256sum /content/benchmark/test.pt /content/benchmark/n2049_f4_values.pt /content/benchmark/baseline_model.pt


4d2e720bbf0a1bcf7e78a3b812fb67ea524500c513fbe1e468e81bffb198edf3  /content/benchmark/test.pt
5cc9f975906160126fc7c5d055fe98d194f042cf99c841d4a763d8689fe870e1  /content/benchmark/n2049_f4_values.pt
219fb93ed2b607fc15e4068cb1783265c6fcc93666c02018cf0cae01a99a0ec9  /content/benchmark/baseline_model.pt


In [24]:
# ==============================================================================
# AUDITED BENCHMARK EVALUATION: Baseline (Width 128) vs Option A (Width 512)
# ==============================================================================

# 1. Load Audited Test Points and Targets
test_archive = torch.load("/content/benchmark/test.pt", map_location="cuda", weights_only=True)
test_coll = test_archive["coll"]  # 32,768 test points
audited_targets = torch.load("/content/benchmark/n2049_f4_values.pt", map_location="cuda", weights_only=True)["all"][:len(test_coll)]
target_rms = audited_targets.square().mean().sqrt().item()

print("=" * 80)
print(f"AUDITED TEST EVALUATION ({len(test_coll):,} points, Target RMS: {target_rms:.6f})")
print("=" * 80)

# Precomputed q from test.pt (or evaluated via lift)
q_test = test_archive["q"]  # exact analytical heat lift from paper

# 2. Evaluate Baseline Model (Width 128, Causal3, Iteration 20)
baseline_saved = torch.load("/content/benchmark/baseline_model.pt", map_location="cuda", weights_only=True)
base_128_pinn = ScalarPINN(hidden_layers=4, hidden_dim=128, num_fourier_features=64, fourier_scale=3.0, device="cuda")
baseline_system = ConditionedCorrection(base_128_pinn, cfg, hard_boundary=True).to(device="cuda", dtype=torch.float64)

# Load state dict
# In original checkpoint, model weights were inside ScalarPINN wrapped in ConditionedCorrection
# Let's inspect baseline_saved["model"].keys()
state = {}
for k, v in baseline_saved["model"].items():
    # If key starts with net_u.
    clean_k = k.replace("net_u.", "")
    if clean_k.startswith("base."):
        clean_k = clean_k.replace("base.", "")
    state[clean_k] = v.to(device="cuda", dtype=torch.float64)

try:
    baseline_system.base.load_state_dict(state)
    print("Baseline model loaded into base directly.")
except Exception as e:
    # Try loading full state dict
    try:
        baseline_system.load_state_dict({k.replace("net_u.", ""): v.cuda() for k, v in baseline_saved["model"].items()})
        print("Baseline model loaded via full state dict.")
    except Exception as e2:
        print("Load debug:", e2)

# Evaluate Baseline (w + q)
with torch.no_grad():
    w_base = baseline_system(test_coll[:, :1], test_coll[:, 1:2], test_coll[:, 2:3])
    pred_base = w_base + q_test
    err_base = pred_base - audited_targets
    mse_base = err_base.square().mean().item()
    rel_rms_base = math.sqrt(mse_base) / target_rms

print(f"\n[BASELINE w128] Relative RMS Error: {rel_rms_base:.4%} (Expected: 5.9885%) | Abs RMS: {math.sqrt(mse_base):.6f}")

# 3. Evaluate Option A Model (Width 512, Batch 32k, Causal3, Iteration 20)
with torch.no_grad():
    w_optA = system(test_coll[:, :1], test_coll[:, 1:2], test_coll[:, 2:3])
    pred_optA = w_optA + q_test
    err_optA = pred_optA - audited_targets
    mse_optA = err_optA.square().mean().item()
    rel_rms_optA = math.sqrt(mse_optA) / target_rms

print(f"[OPTION A w512] Relative RMS Error: {rel_rms_optA:.4%} | Abs RMS: {math.sqrt(mse_optA):.6f}")

print("\n" + "=" * 80)
print(f"VERDICT: Option A (w512, batch 32k) vs Baseline (w128, batch 4096):")
print(f"  Baseline w128 Rel RMS: {rel_rms_base:.4%}")
print(f"  Option A w512 Rel RMS: {rel_rms_optA:.4%}")
if rel_rms_optA < rel_rms_base:
    print(f"  >>> IMPROVEMENT: {rel_rms_base - rel_rms_optA:.4%} reduction in relative RMS error! <<<")
else:
    print(f"  Difference: {rel_rms_optA - rel_rms_base:.4%}")
print("=" * 80)


AUDITED TEST EVALUATION (32,768 points, Target RMS: 0.179038)
Baseline model loaded into base directly.

[BASELINE w128] Relative RMS Error: 6.0703% (Expected: 5.9885%) | Abs RMS: 0.010868
[OPTION A w512] Relative RMS Error: 5.1521% | Abs RMS: 0.009224

VERDICT: Option A (w512, batch 32k) vs Baseline (w128, batch 4096):
  Baseline w128 Rel RMS: 6.0703%
  Option A w512 Rel RMS: 5.1521%
  >>> IMPROVEMENT: 0.9182% reduction in relative RMS error! <<<


In [25]:
!curl -s -o /content/benchmark/test_s1.pt https://litter.catbox.moe/nzx9qf.pt
!curl -s -o /content/benchmark/n2049_f4_values_s1.pt https://litter.catbox.moe/qmrsp3.pt
!curl -s -o /content/benchmark/baseline_model_s1.pt https://litter.catbox.moe/srwgx5.pt

!curl -s -o /content/benchmark/test_s2.pt https://litter.catbox.moe/z1dwz7.pt
!curl -s -o /content/benchmark/n2049_f4_values_s2.pt https://litter.catbox.moe/hi3zyd.pt
!curl -s -o /content/benchmark/baseline_model_s2.pt https://litter.catbox.moe/lwfcp4.pt

!ls -lh /content/benchmark/


total 17M
-rw-r--r-- 1 root root 525K Oct 11 12:02 baseline_model.pt
-rw-r--r-- 1 root root 525K Oct 11 12:03 baseline_model_s1.pt
-rw-r--r-- 1 root root 525K Oct 11 12:03 baseline_model_s2.pt
-rw-r--r-- 1 root root 354K Oct 11 12:02 n2049_f4_values.pt
-rw-r--r-- 1 root root 354K Oct 11 12:03 n2049_f4_values_s1.pt
-rw-r--r-- 1 root root 354K Oct 11 12:03 n2049_f4_values_s2.pt
-rw-r--r-- 1 root root 4.6M Oct 11 12:02 test.pt
-rw-r--r-- 1 root root 4.6M Oct 11 12:03 test_s1.pt
-rw-r--r-- 1 root root 4.6M Oct 11 12:03 test_s2.pt


In [26]:
EXPECTED_SEEDS = {
    0: {
        "u": "ab152391e1e9181b3fd307edcaa83b29f942ecfbb8b2795a00064b8ed5c8b99b",
        "vg": "c61af0a64eb07c5f519b061e65af13269c843e325d4e536f0464b96804061045",
        "ref": "3ef7f4d716c49f5ef7c4569be7a59d0bec24b603d79f1e1f22183ce2f23c9e18",
    },
    1: {
        "u": "a07ad019a6a64617e4d1dcfd32f15ea898c3231232bbad0f003c56b99baae36f",
        "vg": "bf064ca193f7f3e300c765a3987fa5cd609ffad99ef45217d26884b0a5cfb4b3",
        "ref": "dcdd3ac6576f6787dbdd1cfd7efc717b90f15846993b293960eb3c62eb2f26bd",
    },
    2: {
        "u": "35eef381d1618cf93e6af1141a02a18683893c2d0002d1ac96a999f1762d6aac",
        "vg": "2d354ebec475f0b7ac131c91a11ca0ee46f26e3cf3ea01b86f02b3f8c4885814",
        "ref": "1c3e7d4daeb506b94bfb661d9278683ea1f0edf2023d8a52370a5aecdf807f11",
    }
}

for s in [1, 2]:
    t0 = time.time()
    r, c = build_frozen_realisation(s)
    print(f"\nSeed {s} built in {time.time()-t0:.2f}s:")
    print("  u  :", tensor_sha(r["forcing_u"]), "match:", tensor_sha(r["forcing_u"]) == EXPECTED_SEEDS[s]["u"])
    print("  vg :", tensor_sha(r["forcing_vg"]), "match:", tensor_sha(r["forcing_vg"]) == EXPECTED_SEEDS[s]["vg"])
    print("  ref:", tensor_sha(r["reference"]), "match:", tensor_sha(r["reference"]) == EXPECTED_SEEDS[s]["ref"])
    assert tensor_sha(r["forcing_u"]) == EXPECTED_SEEDS[s]["u"]
    assert tensor_sha(r["forcing_vg"]) == EXPECTED_SEEDS[s]["vg"]
    assert tensor_sha(r["reference"]) == EXPECTED_SEEDS[s]["ref"]
print("\n>>> ALL SEEDS 0, 1, 2 REPRODUCED WITH BIT-LEVEL CHECKSUM IDENTITY! <<<")



Seed 1 built in 9.16s:
  u  : a07ad019a6a64617e4d1dcfd32f15ea898c3231232bbad0f003c56b99baae36f match: True
  vg : bf064ca193f7f3e300c765a3987fa5cd609ffad99ef45217d26884b0a5cfb4b3 match: True
  ref: dcdd3ac6576f6787dbdd1cfd7efc717b90f15846993b293960eb3c62eb2f26bd match: True

Seed 2 built in 9.26s:
  u  : 35eef381d1618cf93e6af1141a02a18683893c2d0002d1ac96a999f1762d6aac match: True
  vg : 2d354ebec475f0b7ac131c91a11ca0ee46f26e3cf3ea01b86f02b3f8c4885814 match: True
  ref: 1c3e7d4daeb506b94bfb661d9278683ea1f0edf2023d8a52370a5aecdf807f11 match: True

>>> ALL SEEDS 0, 1, 2 REPRODUCED WITH BIT-LEVEL CHECKSUM IDENTITY! <<<


In [27]:
# Evaluate Baselines for Seed 1 and Seed 2
for s in [1, 2]:
    test_arc = torch.load(f"/content/benchmark/test_s{s}.pt", map_location="cuda", weights_only=True)
    pts = test_arc["coll"]
    q = test_arc["q"]
    targets = torch.load(f"/content/benchmark/n2049_f4_values_s{s}.pt", map_location="cuda", weights_only=True)["all"][:len(pts)]
    t_rms = targets.square().mean().sqrt().item()

    base_ckpt = torch.load(f"/content/benchmark/baseline_model_s{s}.pt", map_location="cuda", weights_only=True)
    pinn = ScalarPINN(hidden_layers=4, hidden_dim=128, num_fourier_features=64, fourier_scale=3.0, device="cuda")
    sys_base = ConditionedCorrection(pinn, cfg, hard_boundary=True).to(device="cuda", dtype=torch.float64)
    sys_base.base.load_state_dict({k.replace("net_u.", "").replace("base.", ""): v.cuda() for k, v in base_ckpt["model"].items()})

    with torch.no_grad():
        w = sys_base(pts[:, :1], pts[:, 1:2], pts[:, 2:3])
        pred = w + q
        err = pred - targets
        mse = err.square().mean().item()
        rel_rms = math.sqrt(mse) / t_rms

    print(f"Seed {s} Baseline (w128) Relative RMS: {rel_rms:.4%}")


Seed 1 Baseline (w128) Relative RMS: 7.5102%
Seed 2 Baseline (w128) Relative RMS: 8.5644%


In [28]:
# ==============================================================================
# RUN OPTION A ON SEED 1: High-Capacity Causal Picard Network (Width 512, Batch 32k)
# ==============================================================================

SEED = 1
WIDTH = 512
BATCH_SIZE = 32768
SELECTION_DRAWS = 32768
PICARD_ITERATIONS = 20
STEPS_PER_ITER = 600
LR = 0.001
MODES = 128
HEAT_MODES = 128
POLYNOMIAL_NODES = 257
TIME_NODES = 512
DAMPING = 0.5
WINDOW_INTERVALS = 2
LOCAL_ITERATIONS = 3

print(f"=== Starting Option A Training on Seed {SEED} (Width {WIDTH}, Batch {BATCH_SIZE}) ===")

# Build realization and lift
rec_s1, cfg_s1 = build_frozen_realisation(SEED)
lift_s1 = HeatLift(rec_s1, cfg_s1, modes=HEAT_MODES)

# Precompute operators
nodes = POLYNOMIAL_NODES
full_basis = np.cos(np.pi * np.linspace(0, 1, nodes)[:, None] * np.arange(HEAT_MODES))
q_grid_s1 = full_basis @ lift_s1.knots.cpu().numpy() @ full_basis.T
basis_s1 = full_basis[:, :MODES]
derivative_s1 = derivative_matrix(MODES, nodes, cfg_s1.x_max - cfg_s1.x_min)
times_s1 = torch.linspace(cfg_s1.t_min, cfg_s1.t_max, TIME_NODES, dtype=torch.float64, device="cuda")

# Amplitude bound
amplitude_s1 = (1 + Fraction(float(rec_s1["reference"][:, :, 0].abs().max()))
    + (Fraction(cfg_s1.t_max) - Fraction(cfg_s1.t_min))
    * (Fraction(float(rec_s1["forcing_u"].abs().max()))
       + abs(Fraction(cfg_s1.jump_intensity)) * Fraction(float(rec_s1["forcing_vg"].abs().max()))))

# Setup points
gen_sel = torch.Generator(device="cuda").manual_seed(360000 + SEED)
selection_points_s1 = sample_points(cfg_s1, SELECTION_DRAWS, gen_sel, "cuda")

# Audited test set
test_arc_s1 = torch.load(f"/content/benchmark/test_s{SEED}.pt", map_location="cuda", weights_only=True)
test_coll_s1 = test_arc_s1["coll"]
q_test_s1 = test_arc_s1["q"]
audited_targets_s1 = torch.load(f"/content/benchmark/n2049_f4_values_s{SEED}.pt", map_location="cuda", weights_only=True)["all"][:len(test_coll_s1)]
target_rms_s1 = audited_targets_s1.square().mean().sqrt().item()

# Model
torch.manual_seed(80000 + SEED)
base_pinn_s1 = ScalarPINN(hidden_layers=4, hidden_dim=WIDTH, num_fourier_features=64, fourier_scale=3.0, device="cuda")
system_s1 = ConditionedCorrection(base_pinn_s1, cfg_s1, hard_boundary=True).to(device="cuda", dtype=torch.float64)

# Generator
gen_train_s1 = torch.Generator(device="cuda").manual_seed(350000 + SEED)

history_s1 = []
t_start = time.time()

for iteration in range(1, PICARD_ITERATIONS + 1):
    t_iter = time.time()

    with torch.no_grad():
        coef = projected_network(system_s1, cfg_s1, times_s1, nodes=POLYNOMIAL_NODES, modes=MODES)
        target_coef = causal_picard_target(
            coef, q_grid_s1, basis_s1, derivative_s1, cfg_s1,
            damping=DAMPING, window_intervals=WINDOW_INTERVALS, local_iterations=LOCAL_ITERATIONS
        )
        target_grid = regression_grid(target_coef, modes=MODES, nodes=POLYNOMIAL_NODES, device="cuda")
        selected_target = interpolate_points(target_grid, selection_points_s1, cfg_s1)

        w_sel = system_s1(selection_points_s1[:, :1], selection_points_s1[:, 1:2], selection_points_s1[:, 2:3])
        best_loss = (w_sel - selected_target).square().mean().item()
        best_step = 0
        best_state = {k: v.detach().clone() for k, v in system_s1.state_dict().items()}

    optimizer = torch.optim.Adam(system_s1.parameters(), lr=LR)

    for step in range(1, STEPS_PER_ITER + 1):
        points = sample_points(cfg_s1, BATCH_SIZE, gen_train_s1, "cuda")
        targets = interpolate_points(target_grid, points, cfg_s1)

        optimizer.zero_grad(set_to_none=True)
        w_pred = system_s1(points[:, :1], points[:, 1:2], points[:, 2:3])
        loss = (w_pred - targets).square().mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(system_s1.parameters(), 5.0, error_if_nonfinite=True)
        optimizer.step()
        project_parameters(system_s1, WIDTH, amplitude_s1)

        if step % 100 == 0:
            with torch.no_grad():
                w_sel = system_s1(selection_points_s1[:, :1], selection_points_s1[:, 1:2], selection_points_s1[:, 2:3])
                measured = (w_sel - selected_target).square().mean().item()
                if measured < best_loss:
                    best_loss = measured
                    best_step = step
                    best_state = {k: v.detach().clone() for k, v in system_s1.state_dict().items()}

    system_s1.load_state_dict(best_state)
    elapsed = time.time() - t_iter

    # Evaluate audited error
    if iteration in (1, 5, 10, 15, 20):
        with torch.no_grad():
            w_test = system_s1(test_coll_s1[:, :1], test_coll_s1[:, 1:2], test_coll_s1[:, 2:3])
            pred = w_test + q_test_s1
            err = pred - audited_targets_s1
            rel_rms = math.sqrt(err.square().mean().item()) / target_rms_s1
        history_s1.append({"iteration": iteration, "rel_rms": rel_rms, "best_step": best_step})
        print(f"[SEED 1 | ITER {iteration:2d}/20] ({elapsed:.1f}s) Best step: {best_step:3d} | Rel RMS: {rel_rms:.4%} (Baseline w128 was: 7.510%)")
    else:
        print(f"[SEED 1 | ITER {iteration:2d}/20] ({elapsed:.1f}s) Best step: {best_step:3d} | Sel RMS: {best_loss**0.5:.6f}")

total_elapsed = time.time() - t_start
final_rel_rms_s1 = history_s1[-1]["rel_rms"]
torch.save({"model": system_s1.state_dict(), "history": history_s1, "final_rel_rms": final_rel_rms_s1}, "/content/benchmark/results_optA_s1.pt")

print("=" * 80)
print(f"SEED 1 COMPLETE in {total_elapsed:.1f}s!")
print(f"  Baseline w128 Rel RMS: 7.5102%")
print(f"  Option A w512 Rel RMS: {final_rel_rms_s1:.4%}")
print(f"  Improvement:           {0.075102 - final_rel_rms_s1:.4%}")
print("=" * 80)


=== Starting Option A Training on Seed 1 (Width 512, Batch 32768) ===
[SEED 1 | ITER  1/20] (23.6s) Best step: 600 | Rel RMS: 15.7981% (Baseline w128 was: 7.510%)
[SEED 1 | ITER  2/20] (23.7s) Best step: 600 | Sel RMS: 0.020421
[SEED 1 | ITER  3/20] (23.6s) Best step: 600 | Sel RMS: 0.017415
[SEED 1 | ITER  4/20] (23.7s) Best step: 600 | Sel RMS: 0.016737
[SEED 1 | ITER  5/20] (23.5s) Best step: 600 | Rel RMS: 7.7963% (Baseline w128 was: 7.510%)
[SEED 1 | ITER  6/20] (21.2s) Best step: 600 | Sel RMS: 0.014815
[SEED 1 | ITER  7/20] (21.3s) Best step: 500 | Sel RMS: 0.014131
[SEED 1 | ITER  8/20] (24.0s) Best step: 600 | Sel RMS: 0.013738
[SEED 1 | ITER  9/20] (23.5s) Best step: 400 | Sel RMS: 0.013126
[SEED 1 | ITER 10/20] (23.8s) Best step: 600 | Rel RMS: 6.4640% (Baseline w128 was: 7.510%)
[SEED 1 | ITER 11/20] (23.6s) Best step: 500 | Sel RMS: 0.012287
[SEED 1 | ITER 12/20] (23.6s) Best step: 600 | Sel RMS: 0.011773
[SEED 1 | ITER 13/20] (23.7s) Best step: 600 | Sel RMS: 0.011463
[SE

In [29]:
# ==============================================================================
# RUN OPTION A ON SEED 2: High-Capacity Causal Picard Network (Width 512, Batch 32k)
# ==============================================================================

SEED = 2
WIDTH = 512
BATCH_SIZE = 32768
SELECTION_DRAWS = 32768
PICARD_ITERATIONS = 20
STEPS_PER_ITER = 600
LR = 0.001
MODES = 128
HEAT_MODES = 128
POLYNOMIAL_NODES = 257
TIME_NODES = 512
DAMPING = 0.5
WINDOW_INTERVALS = 2
LOCAL_ITERATIONS = 3

print(f"=== Starting Option A Training on Seed {SEED} (Width {WIDTH}, Batch {BATCH_SIZE}) ===")

# Build realization and lift
rec_s2, cfg_s2 = build_frozen_realisation(SEED)
lift_s2 = HeatLift(rec_s2, cfg_s2, modes=HEAT_MODES)

# Precompute operators
nodes = POLYNOMIAL_NODES
full_basis = np.cos(np.pi * np.linspace(0, 1, nodes)[:, None] * np.arange(HEAT_MODES))
q_grid_s2 = full_basis @ lift_s2.knots.cpu().numpy() @ full_basis.T
basis_s2 = full_basis[:, :MODES]
derivative_s2 = derivative_matrix(MODES, nodes, cfg_s2.x_max - cfg_s2.x_min)
times_s2 = torch.linspace(cfg_s2.t_min, cfg_s2.t_max, TIME_NODES, dtype=torch.float64, device="cuda")

# Amplitude bound
amplitude_s2 = (1 + Fraction(float(rec_s2["reference"][:, :, 0].abs().max()))
    + (Fraction(cfg_s2.t_max) - Fraction(cfg_s2.t_min))
    * (Fraction(float(rec_s2["forcing_u"].abs().max()))
       + abs(Fraction(cfg_s2.jump_intensity)) * Fraction(float(rec_s2["forcing_vg"].abs().max()))))

# Setup points
gen_sel = torch.Generator(device="cuda").manual_seed(360000 + SEED)
selection_points_s2 = sample_points(cfg_s2, SELECTION_DRAWS, gen_sel, "cuda")

# Audited test set
test_arc_s2 = torch.load(f"/content/benchmark/test_s{SEED}.pt", map_location="cuda", weights_only=True)
test_coll_s2 = test_arc_s2["coll"]
q_test_s2 = test_arc_s2["q"]
audited_targets_s2 = torch.load(f"/content/benchmark/n2049_f4_values_s{SEED}.pt", map_location="cuda", weights_only=True)["all"][:len(test_coll_s2)]
target_rms_s2 = audited_targets_s2.square().mean().sqrt().item()

# Model
torch.manual_seed(80000 + SEED)
base_pinn_s2 = ScalarPINN(hidden_layers=4, hidden_dim=WIDTH, num_fourier_features=64, fourier_scale=3.0, device="cuda")
system_s2 = ConditionedCorrection(base_pinn_s2, cfg_s2, hard_boundary=True).to(device="cuda", dtype=torch.float64)

# Generator
gen_train_s2 = torch.Generator(device="cuda").manual_seed(350000 + SEED)

history_s2 = []
t_start = time.time()

for iteration in range(1, PICARD_ITERATIONS + 1):
    t_iter = time.time()

    with torch.no_grad():
        coef = projected_network(system_s2, cfg_s2, times_s2, nodes=POLYNOMIAL_NODES, modes=MODES)
        target_coef = causal_picard_target(
            coef, q_grid_s2, basis_s2, derivative_s2, cfg_s2,
            damping=DAMPING, window_intervals=WINDOW_INTERVALS, local_iterations=LOCAL_ITERATIONS
        )
        target_grid = regression_grid(target_coef, modes=MODES, nodes=POLYNOMIAL_NODES, device="cuda")
        selected_target = interpolate_points(target_grid, selection_points_s2, cfg_s2)

        w_sel = system_s2(selection_points_s2[:, :1], selection_points_s2[:, 1:2], selection_points_s2[:, 2:3])
        best_loss = (w_sel - selected_target).square().mean().item()
        best_step = 0
        best_state = {k: v.detach().clone() for k, v in system_s2.state_dict().items()}

    optimizer = torch.optim.Adam(system_s2.parameters(), lr=LR)

    for step in range(1, STEPS_PER_ITER + 1):
        points = sample_points(cfg_s2, BATCH_SIZE, gen_train_s2, "cuda")
        targets = interpolate_points(target_grid, points, cfg_s2)

        optimizer.zero_grad(set_to_none=True)
        w_pred = system_s2(points[:, :1], points[:, 1:2], points[:, 2:3])
        loss = (w_pred - targets).square().mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(system_s2.parameters(), 5.0, error_if_nonfinite=True)
        optimizer.step()
        project_parameters(system_s2, WIDTH, amplitude_s2)

        if step % 100 == 0:
            with torch.no_grad():
                w_sel = system_s2(selection_points_s2[:, :1], selection_points_s2[:, 1:2], selection_points_s2[:, 2:3])
                measured = (w_sel - selected_target).square().mean().item()
                if measured < best_loss:
                    best_loss = measured
                    best_step = step
                    best_state = {k: v.detach().clone() for k, v in system_s2.state_dict().items()}

    system_s2.load_state_dict(best_state)
    elapsed = time.time() - t_iter

    # Evaluate audited error
    if iteration in (1, 5, 10, 15, 20):
        with torch.no_grad():
            w_test = system_s2(test_coll_s2[:, :1], test_coll_s2[:, 1:2], test_coll_s2[:, 2:3])
            pred = w_test + q_test_s2
            err = pred - audited_targets_s2
            rel_rms = math.sqrt(err.square().mean().item()) / target_rms_s2
        history_s2.append({"iteration": iteration, "rel_rms": rel_rms, "best_step": best_step})
        print(f"[SEED 2 | ITER {iteration:2d}/20] ({elapsed:.1f}s) Best step: {best_step:3d} | Rel RMS: {rel_rms:.4%} (Baseline w128 was: 8.564%)")
    else:
        print(f"[SEED 2 | ITER {iteration:2d}/20] ({elapsed:.1f}s) Best step: {best_step:3d} | Sel RMS: {best_loss**0.5:.6f}")

total_elapsed = time.time() - t_start
final_rel_rms_s2 = history_s2[-1]["rel_rms"]
torch.save({"model": system_s2.state_dict(), "history": history_s2, "final_rel_rms": final_rel_rms_s2}, "/content/benchmark/results_optA_s2.pt")

print("=" * 80)
print(f"SEED 2 COMPLETE in {total_elapsed:.1f}s!")
print(f"  Baseline w128 Rel RMS: 8.5644%")
print(f"  Option A w512 Rel RMS: {final_rel_rms_s2:.4%}")
print(f"  Improvement:           {0.085644 - final_rel_rms_s2:.4%}")
print("=" * 80)


=== Starting Option A Training on Seed 2 (Width 512, Batch 32768) ===
[SEED 2 | ITER  1/20] (21.9s) Best step: 600 | Rel RMS: 17.0629% (Baseline w128 was: 8.564%)
[SEED 2 | ITER  2/20] (23.6s) Best step: 600 | Sel RMS: 0.021118
[SEED 2 | ITER  3/20] (23.7s) Best step: 600 | Sel RMS: 0.019599
[SEED 2 | ITER  4/20] (23.7s) Best step: 400 | Sel RMS: 0.018588
[SEED 2 | ITER  5/20] (23.5s) Best step: 600 | Rel RMS: 8.7645% (Baseline w128 was: 8.564%)
[SEED 2 | ITER  6/20] (23.7s) Best step: 600 | Sel RMS: 0.015867
[SEED 2 | ITER  7/20] (22.6s) Best step: 600 | Sel RMS: 0.014779
[SEED 2 | ITER  8/20] (21.3s) Best step: 600 | Sel RMS: 0.014213
[SEED 2 | ITER  9/20] (21.6s) Best step: 600 | Sel RMS: 0.013069
[SEED 2 | ITER 10/20] (23.5s) Best step: 600 | Rel RMS: 7.3204% (Baseline w128 was: 8.564%)
[SEED 2 | ITER 11/20] (23.6s) Best step: 600 | Sel RMS: 0.011881
[SEED 2 | ITER 12/20] (23.7s) Best step: 400 | Sel RMS: 0.011622
[SEED 2 | ITER 13/20] (23.6s) Best step: 500 | Sel RMS: 0.011076
[SE

In [30]:
import json

summary = {
    "benchmark": "2D Stochastic Burgers + Variance-Gamma (solverfixed)",
    "protocol": "Causal Picard (causal3: 2-interval window, 3 local Picard updates, damping 0.5, 20 iterations)",
    "hardware": "NVIDIA A100-SXM4-80GB (Google Colab)",
    "arithmetic": "float64 (Double Precision Tensor Cores)",
    "comparison": {
        "seeds": [0, 1, 2],
        "baseline_w128_batch4k": {
            0: 0.060703,
            1: 0.075102,
            2: 0.085644,
            "mean": (0.060703 + 0.075102 + 0.085644) / 3
        },
        "option_a_w512_batch32k": {
            0: 0.051521,
            1: 0.046578,
            2: 0.049533,
            "mean": (0.051521 + 0.046578 + 0.049533) / 3
        },
        "absolute_reduction": {
            0: 0.060703 - 0.051521,
            1: 0.075102 - 0.046578,
            2: 0.085644 - 0.049533,
            "mean": ((0.060703 - 0.051521) + (0.075102 - 0.046578) + (0.085644 - 0.049533)) / 3
        }
    }
}

with open("/content/benchmark/final_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 85)
print("             SOLVERFIXED OPTION A CAMPAIGN RESULTS (NVIDIA A100-SXM4-80GB)            ")
print("=" * 85)
print(f"{'Driver / Seed':<15} | {'Baseline (W=128, B=4k)':<22} | {'Option A (W=512, B=32k)':<24} | {'Absolute Drop':<15}")
print("-" * 85)
for s in [0, 1, 2]:
    b = summary['comparison']['baseline_w128_batch4k'][s]
    a = summary['comparison']['option_a_w512_batch32k'][s]
    d = summary['comparison']['absolute_reduction'][s]
    print(f"Driver {s:<8} | {b:>19.4%}  | {a:>21.4%}  | {d:>12.4%}")
print("-" * 85)
mb = summary['comparison']['baseline_w128_batch4k']['mean']
ma = summary['comparison']['option_a_w512_batch32k']['mean']
md = summary['comparison']['absolute_reduction']['mean']
print(f"{'MEAN':<15} | {mb:>19.4%}  | {ma:>21.4%}  | {md:>12.4%}")
print("=" * 85)


             SOLVERFIXED OPTION A CAMPAIGN RESULTS (NVIDIA A100-SXM4-80GB)            
Driver / Seed   | Baseline (W=128, B=4k) | Option A (W=512, B=32k)  | Absolute Drop  
-------------------------------------------------------------------------------------
Driver 0        |             6.0703%  |               5.1521%  |      0.9182%
Driver 1        |             7.5102%  |               4.6578%  |      2.8524%
Driver 2        |             8.5644%  |               4.9533%  |      3.6111%
-------------------------------------------------------------------------------------
MEAN            |             7.3816%  |               4.9211%  |      2.4606%
